In [1]:
from google.colab import files
uploaded=files.upload()

Saving kaggle.json to kaggle.json


In [2]:
import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [3]:
!kaggle datasets download -d mysarahmadbhat/airbnb-listings-reviews

Dataset URL: https://www.kaggle.com/datasets/mysarahmadbhat/airbnb-listings-reviews
License(s): CC0-1.0
100% 101M/101M [00:01<00:00, 88.5MB/s] 



In [4]:
!unzip -q airbnb-listings-reviews.zip -d airbnb_data

In [5]:
import os
os.listdir('airbnb_data')

['Airbnb Data']

In [6]:
for f in os.listdir('airbnb_data'):
    size_mb = os.path.getsize(f'airbnb_data/{f}') / (1024*1024)
    print(f, round(size_mb, 2), "MB")


Airbnb Data 0.0 MB


In [7]:
os.listdir('airbnb_data/Airbnb Data')

['Reviews.csv',
 'Listings.csv',
 'Reviews_data_dictionary.csv',
 'Listings_data_dictionary.csv']

In [8]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("airbnb.db")

# encoding='latin1' avoids a UnicodeDecodeError — this file has some special characters in it
listings = pd.read_csv("airbnb_data/Airbnb Data/Listings.csv", encoding="latin1")
reviews = pd.read_csv("airbnb_data/Airbnb Data/Reviews.csv", encoding="latin1")

listings.to_sql("listings", conn, if_exists="replace", index=False)
reviews.to_sql("reviews", conn, if_exists="replace", index=False)

print("Listings:", len(listings), "rows loaded")
print("Reviews:", len(reviews), "rows loaded")

/tmp/ipykernel_4018/1132514648.py:7: DtypeWarning: Columns (5,13) have mixed types. Specify dtype option on import or set low_memory=False.
  listings = pd.read_csv("airbnb_data/Airbnb Data/Listings.csv", encoding="latin1")


Listings: 279712 rows loaded
Reviews: 5373143 rows loaded


In [9]:
query="PRAGMA table_info(listings)"
pd.read_sql_query(query, conn)

,cid,name,type,notnull,dflt_value,pk
0,0,listing_id,INTEGER,0,None,0
1,1,name,TEXT,0,None,0
2,2,host_id,INTEGER,0,None,0
3,3,host_since,TEXT,0,None,0
4,4,host_location,TEXT,0,None,0
5,5,host_response_time,TEXT,0,None,0
6,6,host_response_rate,REAL,0,None,0
7,7,host_acceptance_rate,REAL,0,None,0
8,8,host_is_superhost,TEXT,0,None,0
9,9,host_total_listings_count,REAL,0,None,0


In [10]:
query="PRAGMA table_info(reviews)"
pd.read_sql_query(query,conn)

,cid,name,type,notnull,dflt_value,pk
0,0,listing_id,INTEGER,0,None,0
1,1,review_id,INTEGER,0,None,0
2,2,date,TEXT,0,None,0
3,3,reviewer_id,INTEGER,0,None,0


In [11]:
query="""
select *
from listings
limit 5
"""
pd.read_sql_query(query,conn)

,listing_id,name,host_id,host_since,host_location,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_total_listings_count,...,minimum_nights,maximum_nights,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable
0,281420,"Beautiful Flat in le Village Montmartre, Paris",1466919,2011-12-03,"Paris, Ile-de-France, France",None,None,None,f,1.0,...,2,1125,100.0,10.0,10.0,10.0,10.0,10.0,10.0,f
1,3705183,39 mÃÂ² Paris (Sacre CÃ âur),10328771,2013-11-29,"Paris, Ile-de-France, France",None,None,None,f,1.0,...,2,1125,100.0,10.0,10.0,10.0,10.0,10.0,10.0,f
2,4082273,"Lovely apartment with Terrace, 60m2",19252768,2014-07-31,"Paris, Ile-de-France, France",None,None,None,f,1.0,...,2,1125,100.0,10.0,10.0,10.0,10.0,10.0,10.0,f
3,4797344,Cosy studio (close to Eiffel tower),10668311,2013-12-17,"Paris, Ile-de-France, France",None,None,None,f,1.0,...,2,1125,100.0,10.0,10.0,10.0,10.0,10.0,10.0,f
4,4823489,Close to Eiffel Tower - Beautiful flat : 2 rooms,24837558,2014-12-14,"Paris, Ile-de-France, France",None,None,None,f,1.0,...,2,1125,100.0,10.0,10.0,10.0,10.0,10.0,10.0,f


In [12]:
query = """
SELECT COUNT(*) AS total_listings
FROM listings;
"""
pd.read_sql_query(query, conn)

,total_listings
0,279712


In [13]:
query = """
SELECT DISTINCT city
FROM listings;
"""
pd.read_sql_query(query, conn)

,city
0,Paris
1,New York
2,Bangkok
3,Rio de Janeiro
4,Sydney
5,Istanbul
6,Rome
7,Hong Kong
8,Mexico City
9,Cape Town


In [14]:
query="""
select city, count(*) as num_listings
from listings
group by city
order by num_listings desc;
"""
pd.read_sql_query(query,conn)

,city,num_listings
0,Paris,64690
1,New York,37012
2,Sydney,33630
3,Rome,27647
4,Rio de Janeiro,26615
5,Istanbul,24519
6,Mexico City,20065
7,Bangkok,19361
8,Cape Town,19086
9,Hong Kong,7087


In [15]:
query="""
select listing_id,count(*) as cnt
from listings
group by listing_id
having count(*)>1;
"""
pd.read_sql_query(query,conn)

,listing_id,cnt


In [16]:
query="""
select
sum(case when district is null then 1 else 0 end) as null_district,
sum(case when host_response_rate is null then 1 else 0 end) as null_host_response_rate,
sum(case when review_scores_rating is null then 1 else 0 end) as null_review_scores_rating
from listings;
"""
pd.read_sql_query(query,conn)

,null_district,null_host_response_rate,null_review_scores_rating
0,242700,128782,91405


In [17]:
conn.execute("drop table if exists listings_clean")
conn.commit()

query="""
create table listings_clean as
select * from listings;
"""
conn.execute(query)
conn.commit()

In [18]:
pd.read_sql_query("select count(*) as clean_rows from listings_clean",conn)

,clean_rows
0,279712


In [19]:
# Question 1: Average price and listing count per city
# Why: this is the first question any business stakeholder would ask — "where are we biggest, and how do prices compare across cities?"

query="""
select city,
count(*) as num_listings,
round(avg(price),2) as avg_price
from listings_clean
where price is not null
group by city
order by avg_price desc;
"""
pd.read_sql_query(query,conn)


,city,num_listings,avg_price
0,Cape Town,19086,2405.12
1,Bangkok,19361,2078.28
2,Mexico City,20065,1149.25
3,Hong Kong,7087,746.17
4,Rio de Janeiro,26615,742.59
5,Istanbul,24519,532.56
6,Sydney,33630,222.01
7,New York,37012,142.84
8,Paris,64690,113.10
9,Rome,27647,105.11


In [20]:
# Question 2: Which room types are most common?
# Why: room type is one of Airbnb's core product categories — entire homes vs private rooms vs shared rooms. Understanding the split tells you what kind of inventory dominates the platform, and it's a classic "distribution" question interviewers love to ask analysts.

query="""
select room_type,
count(*) as num_listings,
round(avg(price),2) as avg_price
from listings_clean
where price is not null
group by room_type
order by avg_price desc;
"""
pd.read_sql_query(query,conn)

,room_type,num_listings,avg_price
0,Hotel room,5857,800.21
1,Entire place,182005,673.35
2,Shared room,4862,579.92
3,Private room,86988,462.44


In [21]:
# Question 3: Which hosts own the most listings?
# Why: identifying "power hosts" — hosts who manage multiple properties like a mini hotel business — is a real Airbnb business question. It also tells you how concentrated the supply is (are a few hosts dominating, or is it spread evenly?). This introduces HAVING in a practical, meaningful context.

query="""
select host_id, host_since, count(*) as total_listings
from listings_clean
group by host_id, host_since
having count(*)>=10
order by total_listings desc
limit 15;
"""
pd.read_sql_query(query, conn)

,host_id,host_since,total_listings
0,291007369,2019-09-02,627
1,175128252,2018-02-24,396
2,7518056,2013-07-16,378
3,97240131,2016-09-29,339
4,371026651,2020-10-07,295
5,4584648,2013-01-04,282
6,107434423,2016-12-16,255
7,6053288,2013-04-23,241
8,2667370,2012-06-18,236
9,33889201,2015-05-21,215


In [22]:
# Question 4: Average rating per city — which city has the happiest guests?
# Why: rating is the single most important trust signal on Airbnb — it drives bookings. Comparing average ratings across cities tells you where guest satisfaction is highest, and where Airbnb might need to focus quality improvements. Classic business question.

query="""
select city,
round(avg(review_scores_rating),1) as avg_rating,
count(*) as listings_with_ratings
from listings_clean
where review_scores_rating is not null
group by city
order by avg_rating desc;
"""

pd.read_sql_query(query,conn)

,city,avg_rating,listings_with_ratings
0,Mexico City,94.8,14484
1,Rio de Janeiro,94.6,16118
2,Cape Town,94.4,13435
3,New York,93.8,26777
4,Rome,93.5,20893
5,Sydney,93.2,22376
6,Paris,93.1,48036
7,Bangkok,93.0,11181
8,Istanbul,91.1,11229
9,Hong Kong,89.7,3778


In [23]:
# Question 5: Most expensive neighbourhoods in New York
# Why: city-level pricing is too broad for real decisions — a guest or business analyst needs neighbourhood-level granularity. New York is the best city for this because it's USD-priced (no currency confusion) and has globally recognisable neighbourhoods (Manhattan, Brooklyn etc.) — making the output immediately understandable to any interviewer. This also introduces HAVING in a more meaningful, realistic way than just duplicate-checking.

query="""
select neighbourhood,
round(avg(price),2) as avg_price,
count(*) as num_listings
from listings_clean
where city="New York" and price is not null
group by neighbourhood
having count(*)>=10
order by avg_price desc
limit 10;
"""

pd.read_sql_query(query,conn)


,neighbourhood,avg_price,num_listings
0,Tribeca,373.20,167
1,Briarwood,346.15,34
2,Flatiron District,324.17,75
3,Sea Gate,270.91,11
4,SoHo,265.85,282
5,Theater District,262.25,303
6,Greenwich Village,254.40,259
7,Lower East Side,249.52,707
8,Fieldston,243.33,12
9,Arverne,232.02,64


In [24]:
# Question 6: First JOIN — how many reviews has each listing received?

query="""
select l.listing_id,
l.name,
l.city,
l.price,
count(r.review_id) as total_reviews
from listings_clean as l
left join reviews as r
on l.listing_id = r.listing_id
group by l.listing_id,l.name,l.city,l.price
order by total_reviews desc
limit 10;
"""

pd.read_sql_query(query,conn)

,listing_id,name,city,price,total_reviews
0,17222007,Sweet & cosy room next to Canal Saint Martin Ã...,Paris,76,891
1,8637229,Ã©Â¦â¢Ã¦Â¸Â¯ Hong Kong Ã©â¢Â·Ã¦Â´Â² Cheung C...,Hong Kong,629,828
2,1249964,A Journey Into The Heart Of Paris,Paris,81,796
3,32011332,Bed in 8 Bed Dorm,Paris,27,762
4,2399029,B&B Rione Monti,Rome,39,754
5,32678719,Enjoy great views of the City with FREE PARKING!,New York,77,753
6,470817,Caroline's cheap rate near Termini,Rome,35,717
7,24745583,Bed in a 4 Bed Mixed Dormitory within hostel,Paris,36,710
8,865289,Colonna's Home in the heart of Rome,Rome,26,702
9,162163,Brand new Modern flat Central Paris Boutique H...,Paris,96,700


In [25]:
# Question 7: Listings with most reviews but below-average rating
# Why: this is the most business-intelligent question so far. A listing with hundreds of reviews but a low rating is a red flag — it means many guests had a bad experience. Airbnb would want to identify these and potentially delist or warn them.

query="""
select l.listing_id,
l.name,
l.city,
l.review_scores_rating,
count(r.review_id) as total_reviews
from listings_clean as l
left join reviews as r
on l.listing_id = r.listing_id
where l.review_scores_rating <(select avg(review_scores_rating) from listings_clean where review_scores_rating is not null)
and l.review_scores_rating is not null
group by l.listing_id,l.name,l.city,l.review_scores_rating
order by total_reviews desc
limit 10;
"""

pd.read_sql_query(query,conn)

,listing_id,name,city,review_scores_rating,total_reviews
0,32011332,Bed in 8 Bed Dorm,Paris,87.0,762
1,2399029,B&B Rione Monti,Rome,93.0,754
2,32678719,Enjoy great views of the City with FREE PARKING!,New York,85.0,753
3,470817,Caroline's cheap rate near Termini,Rome,86.0,717
4,24745583,Bed in a 4 Bed Mixed Dormitory within hostel,Paris,86.0,710
5,865289,Colonna's Home in the heart of Rome,Rome,85.0,702
6,162163,Brand new Modern flat Central Paris Boutique H...,Paris,86.0,700
7,2488829,Place de la Bastille.,Paris,92.0,636
8,5724631,Amazing view of the Eiffel Tower!,Paris,90.0,621
9,35065,Lovely Loft Saint-Germain des Pres,Paris,93.0,608


In [26]:
# Window Query 1: Top 3 most expensive listings per city

query="""
WITH ranked_listings AS(
select
listing_id,
name,
city,
price,
ROW_NUMBER()OVER(PARTITION BY city ORDER BY price) AS price_rank
from listings_clean
where price is not null
)
select *
from ranked_listings
where price_rank<=3
order by city, price_rank;
"""

pd.read_sql_query(query,conn)


,listing_id,name,city,price,price_rank
0,44563108,Somerset Maison Asoke Bangkok,Bangkok,0,1
1,48072226,Boss room BKK 121 NEAR MRT BL04+1bath+pocket wifi,Bangkok,241,2
2,48075076,Boss room BKK 122 NEAR MRT BL04+1bath+pocket wifi,Bangkok,241,3
3,46692010,Affordable beds in Awesome location,Cape Town,122,1
4,48086756,10 Diedrikkie Rd,Cape Town,124,2
5,45935299,Student Backpacker Accommodation in Obs,Cape Town,129,3
6,43345284,Hong Kong Ã§Åâ¹Ã¥Â¾âÃ¨Â¦â¹Ã©Â¢Â¨Ã¦â¢Â¯Ã§...,Hong Kong,1,1
7,47932081,Ã¤Â¸â¡Ã¨Â±Â¡Ã¥Å¸Å½Ã¦âÂÃ¥Â¤ÂÃ¥Â¼ÂÃ§Â²Â¾Ã¨...,Hong Kong,65,2
8,48238640,Ã¦ÅâÃ¥â¦ÆÃ©Â¾ÂÃ¨Â¡â9Ã¨â¢Å¸Ã¦âÂ°Ã¦â...,Hong Kong,74,3
9,43425821,Armada Oldcity Hotel,Istanbul,0,1


In [27]:
# Window Query 2: Rank listings within each city by rating

query = """
WITH rating_ranks AS (
    SELECT
        listing_id,
        name,
        city,
        review_scores_rating,
        price,
        ROW_NUMBER() OVER (
            PARTITION BY city
            ORDER BY review_scores_rating DESC, price DESC
        ) AS rating_rank
    FROM listings_clean
    WHERE review_scores_rating IS NOT NULL
)
SELECT *
FROM rating_ranks
WHERE rating_rank <= 3
ORDER BY city, rating_rank;
"""
pd.read_sql_query(query, conn)

,listing_id,name,city,review_scores_rating,price,rating_rank
0,37227373,Silom-Sathorn Near bts/mrt 43 sq.m private room,Bangkok,100.0,99000,1
1,15742265,Quiet Canal Oasis Apartment very near BTS,Bangkok,100.0,98000,2
2,17980282,"Superior,5min walk MRT Sukhumvit,WiFi,Terminal21",Bangkok,100.0,50400,3
3,19693180,Budget Reception Room,Cape Town,100.0,69213,1
4,5391764,Luxus- The Lap of Luxury,Cape Town,100.0,65357,2
5,21571263,"Clifton Luxury Villa, Cape Town",Cape Town,100.0,65000,3
6,29890300,Causeway Bay!Ã¦Â­Â¥Ã¨Â¡ÅÃ§ÂºÂ¦10Ã¥Ëâ Ã©âÅ...,Hong Kong,100.0,66666,1
7,16670700,Ã¨Â¥Â¿Ã¤Â¹ÂÃ©Â¾ÂÃ¤Â¸Â­Ã¥Â¿ÆÃ©â¢âÃ¨Â¿âÃ...,Hong Kong,100.0,35000,2
8,47182436,Castle Villa Ã¨Â±ÂªÃ¨ÂÂ¯Ã¥ÂÂ¤Ã¥Â Â¡Ã¥Â¼ÂÃ¥Ë...,Hong Kong,100.0,31000,3
9,16401723,perfec location,Istanbul,100.0,179153,1


In [28]:
# Window Query 3: Month-over-month review growth using LAG
# Why: this is the most different window function from what you've seen. Instead of ranking rows, LAG lets you look back at the previous row — which is how you compare "this month vs last month.

query="""
WITH monthly_reviews AS(
select
STRFTIME('%Y-%m', date) AS month,
count(*) as total_reviews
from reviews
group by month
),

with_lag AS(
select
month,
total_reviews,
lag(total_reviews) over(order by month) as prev_month_reviews
from monthly_reviews
)
select
month,
total_reviews,
prev_month_reviews,
total_reviews-prev_month_reviews as growth
from with_lag
where prev_month_reviews is not null
order by month
limit 24;
"""

pd.read_sql_query(query,conn)

,month,total_reviews,prev_month_reviews,growth
0,2009-01,1,2,-1
1,2009-02,1,1,0
2,2009-04,2,1,1
3,2009-05,6,2,4
4,2009-06,6,6,0
5,2009-07,14,6,8
6,2009-08,7,14,-7
7,2009-09,10,7,3
8,2009-10,16,10,6
9,2009-11,24,16,8


In [29]:
# Window Query 3: Month-over-month review growth using LAG
# Why: this is the most different window function from what you've seen. Instead of ranking rows, LAG lets you look back at the previous row — which is how you compare "this month vs last month.

query="""
WITH monthly_reviews AS(
select
STRFTIME('%Y-%m', date) AS month,
count(*) as total_reviews
from reviews
group by month
),

with_lag AS(
select
month,
total_reviews,
lag(total_reviews) over(order by month) as prev_month_reviews
from monthly_reviews
)
select
month,
total_reviews,
prev_month_reviews,
total_reviews-prev_month_reviews as growth
from with_lag
where prev_month_reviews is not null
order by month
limit 24;
"""

pd.read_sql_query(query,conn)

,month,total_reviews,prev_month_reviews,growth
0,2009-01,1,2,-1
1,2009-02,1,1,0
2,2009-04,2,1,1
3,2009-05,6,2,4
4,2009-06,6,6,0
5,2009-07,14,6,8
6,2009-08,7,14,-7
7,2009-09,10,7,3
8,2009-10,16,10,6
9,2009-11,24,16,8


In [30]:
# Window Query 4: Running total of reviews over time
# Why: a running total shows you the cumulative growth of the platform over time — not just "how many reviews this month" but "how many total reviews have ever been posted up to this point." This is a classic business metric (cumulative users, cumulative revenue, cumulative bookings)
query="""
WITH monthly_reviews AS(
select
STRFTIME('%Y-%m',date) as month,
count(*) as total_reviews
from reviews
group by month
)
select
month,
total_reviews,
sum(total_reviews) over(order by month) as running_total
from monthly_reviews
order by month
limit 24;
"""

pd.read_sql_query(query,conn)

,month,total_reviews,running_total
0,2008-11,2,2
1,2009-01,1,3
2,2009-02,1,4
3,2009-04,2,6
4,2009-05,6,12
5,2009-06,6,18
6,2009-07,14,32
7,2009-08,7,39
8,2009-09,10,49
9,2009-10,16,65
